In [2]:
import torch
import numpy as np
from pathlib import Path


In [3]:
print(torch.__version__)
print(torch.version.hip)

print(
    torch.cuda.is_available()
)

if torch.cuda.is_available():
    print(
        torch.cuda.get_device_name(0)
    )

2.13.0+rocm10.0.0
7.15.26333
True
AMD Radeon(TM) 8060S Graphics


In [4]:
Root = Path.cwd()

Sample_Path = (Root/"output"/"samples"/"sample_video_tensor.pt")


Video_Tensor = torch.load(
    Sample_Path
)


print(Video_Tensor.shape)
print(Video_Tensor.dtype)

torch.Size([1, 3, 16, 224, 224])
torch.float32


In [5]:
Device = "cuda"

Video_Tensor = Video_Tensor.to(Device)
print(f"{Video_Tensor.device}")

cuda:0


In [6]:
from transformers import AutoModel


MODEL_ID = "galilai-group/LeVJEPA-VideoMix-Large"


model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code = True, use_safetensors = True)


print("Model loaded successfully")

c:\Users\Mahed\.conda\envs\levjepa\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 293/293 [00:00<00:00, 5675.65it/s]

Model loaded successfully


In [7]:
model = model.to(Device)

model.eval()

LeVJEPAModel(
  (encoder): VisionTransformer(
    (patch_embed): PatchEmbed3D(
      (proj): Conv3d(3, 1024, kernel_size=(1, 16, 16), stride=(1, 16, 16))
    )
    (blocks): ModuleList(
      (0-23): 24 x Block(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
        (attn): RoPEAttention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
        (mlp): MLP(
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((1024,), eps=1e-06, elementwis

In [8]:
print(next(model.parameters()).device)

cuda:0


In [10]:
model.eval()

with torch.inference_mode():
    Model_Output = model(
        pixel_values=Video_Tensor
    )


In [11]:
print(
    "Output Type:",
    type(Model_Output)
)

print(
    "Output Keys:",
    Model_Output.keys()
)

print(
    "Last Hidden State Shape:",
    Model_Output.last_hidden_state.shape
)

print(
    "Dtype:",
    Model_Output.last_hidden_state.dtype
)

print(
    "Device:",
    Model_Output.last_hidden_state.device
)

Output Type: <class 'transformers.modeling_outputs.BaseModelOutput'>
Output Keys: odict_keys(['last_hidden_state', 'pooler_output'])
Last Hidden State Shape: torch.Size([1, 3137, 1024])
Dtype: torch.float32
Device: cuda:0


In [12]:
Last_Hidden_State = Model_Output.last_hidden_state

CLS_Token = Last_Hidden_State[:, 0, :]

Patch_Tokens = Last_Hidden_State[:, 1:, :]


print(
    "CLS Token Shape:",
    CLS_Token.shape
)

print(
    "Patch Tokens Shape:",
    Patch_Tokens.shape
)

CLS Token Shape: torch.Size([1, 1024])
Patch Tokens Shape: torch.Size([1, 3136, 1024])


In [13]:
Expected_Patch_Tokens = (
    16
    *
    14
    *
    14
)

print(
    "Expected Patch Tokens:",
    Expected_Patch_Tokens
)

print(
    "Actual Patch Tokens:",
    Patch_Tokens.shape[1]
)

assert Patch_Tokens.shape[1] == Expected_Patch_Tokens

print(
    "Patch token validation passed."
)

Expected Patch Tokens: 3136
Actual Patch Tokens: 3136
Patch token validation passed.


In [14]:
Patch_Features = Patch_Tokens.reshape(
    1,
    16,
    14,
    14,
    1024
)

print(
    "Patch Features Shape:",
    Patch_Features.shape
)

Patch Features Shape: torch.Size([1, 16, 14, 14, 1024])


In [15]:
print(
    "Patch Features Shape:",
    Patch_Features.shape
)

print(
    "Dtype:",
    Patch_Features.dtype
)

print(
    "Device:",
    Patch_Features.device
)

print(
    "Finite:",
    torch.isfinite(Patch_Features).all().item()
)

Patch Features Shape: torch.Size([1, 16, 14, 14, 1024])
Dtype: torch.float32
Device: cuda:0
Finite: True


In [16]:
First_Frame_Features = Patch_Features[
    0,
    0
]

print(
    "First Frame Features Shape:",
    First_Frame_Features.shape
)

First Frame Features Shape: torch.Size([14, 14, 1024])


In [17]:
Center_Patch_Feature = First_Frame_Features[
    7,
    7
]

print(
    "Center Patch Shape:",
    Center_Patch_Feature.shape
)

print(
    "First 10 values:",
    Center_Patch_Feature[:10]
)

Center Patch Shape: torch.Size([1024])
First 10 values: tensor([-0.3261, -0.4880, -0.2485,  0.3940, -0.4395, -0.6499,  0.2945,  0.6648,
         0.2721,  0.1830], device='cuda:0')


In [18]:
Pooler_Output = Model_Output.pooler_output

print(
    "Pooler Output Shape:",
    Pooler_Output.shape
)

print(
    "Dtype:",
    Pooler_Output.dtype
)

print(
    "Device:",
    Pooler_Output.device
)

Pooler Output Shape: torch.Size([1, 1024])
Dtype: torch.float32
Device: cuda:0


In [19]:
Features_Dir = (
    Root
    / "output"
    / "features"
)

Features_Dir.mkdir(
    parents=True,
    exist_ok=True
)

Feature_Path = (
    Features_Dir
    / "sample_video_features.pt"
)

In [20]:
torch.save(
    {
        "model_id": MODEL_ID,
        "input_shape": tuple(Video_Tensor.shape),

        "cls_token": (
            CLS_Token
            .detach()
            .cpu()
        ),

        "pooler_output": (
            Pooler_Output
            .detach()
            .cpu()
        ),

        "patch_features": (
            Patch_Features
            .detach()
            .cpu()
        )
    },
    Feature_Path
)

print(
    "Features saved at:",
    Feature_Path
)

Features saved at: d:\project\Personal\Hadi\EchoJEPA\output\features\sample_video_features.pt


In [21]:
Saved_Features = torch.load(
    Feature_Path,
    map_location="cpu"
)

print(
    "CLS Token:",
    Saved_Features["cls_token"].shape
)

print(
    "Pooler Output:",
    Saved_Features["pooler_output"].shape
)

print(
    "Patch Features:",
    Saved_Features["patch_features"].shape
)

print(
    "LeVJEPA feature extraction completed successfully."
)

CLS Token: torch.Size([1, 1024])
Pooler Output: torch.Size([1, 1024])
Patch Features: torch.Size([1, 16, 14, 14, 1024])
LeVJEPA feature extraction completed successfully.
